In [ ]:
#Programming Exercise 9-1 MuZero-Style Network
# Your task is to complete the missing components in cell 3so that the network works correctly.

In [ ]:
# Cell 1: Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [ ]:
# Cell 2: Simple Environment (Gridworld)
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.reset()

    def reset(self):
        self.agent = [0, 0]
        self.goal = [self.size-1, self.size-1]
        return self._get_state()

    def _get_state(self):
        state = np.zeros((self.size, self.size))
        state[self.agent[0], self.agent[1]] = 1
        return state.flatten()

    def step(self, action):
        # 0=up, 1=down, 2=left, 3=right
        if action == 0 and self.agent[0] > 0:
            self.agent[0] -= 1
        elif action == 1 and self.agent[0] < self.size-1:
            self.agent[0] += 1
        elif action == 2 and self.agent[1] > 0:
            self.agent[1] -= 1
        elif action == 3 and self.agent[1] < self.size-1:
            self.agent[1] += 1

        reward = 1 if self.agent == self.goal else -0.01
        done = self.agent == self.goal
        return self._get_state(), reward, done

In [ ]:
# Cell 3: MuZero-Style Network (COMPLETE THE MISSING PARTS)

class MuZeroNet(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super().__init__()

        # =========================
        # 1. Representation function
        # =========================
        # TODO: Create a neural network that maps:
        # state_size → hidden_size → ReLU
        self.representation = nn.Sequential(
            # FILL IN: Linear layer (state_size → hidden_size)
            nn.Linear(____, ____),
            nn.ReLU()
        )

        # =========================
        # 2. Dynamics function
        # =========================
        # Input: hidden_size + action_size
        # Output: hidden_size → ReLU
        self.dynamics = nn.Sequential(
            # FILL IN: Linear layer (hidden_size + action_size → hidden_size)
            nn.Linear(____, ____),
            nn.ReLU()
        )

        # =========================
        # 3. Reward prediction head
        # =========================
        # Input: hidden_size → Output: 1
        self.reward_head = nn.Linear(____, ____)

        # =========================
        # 4. Policy prediction head
        # =========================
        # Input: hidden_size → Output: action_size
        self.policy_head = nn.Linear(____, ____)

        # =========================
        # 5. Value prediction head
        # =========================
        # Input: hidden_size → Output: 1
        self.value_head = nn.Linear(____, ____)

    # ==========================================================
    # 6. Initial inference (state → hidden, policy, value)
    # ==========================================================
    def initial_inference(self, state):
        # TODO: compute hidden representation
        hidden = self.representation(____)

        # TODO: compute policy logits from hidden
        policy = self.policy_head(____)

        # TODO: compute value estimate from hidden
        value = self.value_head(____)

        return hidden, policy, value

    # ==========================================================
    # 7. Recurrent inference (hidden + action → next prediction)
    # ==========================================================
    def recurrent_inference(self, hidden, action_onehot):
        # Step 1: concatenate hidden + action
        x = torch.cat([____, ____], dim=1)

        # Step 2: predict next hidden state
        next_hidden = self.dynamics(____)

        # Step 3: predict reward from next hidden state
        reward = self.reward_head(____)

        # Step 4: predict policy from next hidden state
        policy = self.policy_head(____)

        # Step 5: predict value from next hidden state
        value = self.value_head(____)

        return next_hidden, reward, policy, value

In [ ]:
# Cell 4: Training Setup
env = GridWorld()
state_size = 25
action_size = 4

model = MuZeroNet(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def one_hot(action, size):
    vec = np.zeros(size)
    vec[action] = 1
    return vec

In [ ]:
# Cell 5: Training Loop
episodes = 50

for ep in range(episodes):
    state = env.reset()
    state = torch.FloatTensor(state).unsqueeze(0)

    total_reward = 0

    for step in range(50):
        hidden, policy, value = model.initial_inference(state)

        action = np.random.choice(4)  # random exploration
        next_state, reward, done = env.step(action)

        next_state_t = torch.FloatTensor(next_state).unsqueeze(0)
        action_oh = torch.FloatTensor(one_hot(action, 4)).unsqueeze(0)

        next_hidden, pred_reward, pred_policy, pred_value = model.recurrent_inference(hidden, action_oh)

        # Targets
        target_reward = torch.tensor([[reward]], dtype=torch.float32)
        target_value = torch.tensor([[reward]], dtype=torch.float32)

        # Loss
        loss = (
            nn.MSELoss()(pred_reward, target_reward) +
            nn.MSELoss()(pred_value, target_value)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state_t
        total_reward += reward

        if done:
            break

    print(f"Episode {ep+1}: Total Reward = {total_reward:.2f}")

In [ ]:
# Cell 6: Inspect Learned Behavior
state = env.reset()
state = torch.FloatTensor(state).unsqueeze(0)

# Initial inference (no action yet)
hidden, policy, value = model.initial_inference(state)

print("=== INITIAL PREDICTION (Before Taking Action) ===")
print("Policy (action probabilities):")
print(torch.softmax(policy, dim=1).detach().numpy())

print("\n[KEY IDEA] The model predicts VALUE, not just actions:")
print(f"Estimated Value of Current State: {value.item():.4f}")

# Take a sample action to demonstrate dynamics
action = np.argmax(torch.softmax(policy, dim=1).detach().numpy())
action_oh = torch.FloatTensor(one_hot(action, 4)).unsqueeze(0)

next_hidden, pred_reward, next_policy, next_value = model.recurrent_inference(hidden, action_oh)

print("\n=== AFTER TAKING ACTION (Using Dynamics Function) ===")
print(f"Chosen Action: {action}")

print("\n[KEY IDEA] The model predicts REWARD from the action:")
print(f"Predicted Reward: {pred_reward.item():.4f}")

print("\n[KEY IDEA] The dynamics function simulates the next state internally:")
print("Next Policy (from imagined next state):")
print(torch.softmax(next_policy, dim=1).detach().numpy())

print("\nNext State Value Estimate:")
print(f"{next_value.item():.4f}")

print("\n=== LEARNING PROGRESS NOTE ===")
print("[KEY IDEA] Over time, total rewards printed during training should improve.")
print("This indicates the model is learning the environment structure.")